In [13]:
#Installments
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn
!pip install scikit-learn
!pip install imbalanced-learn
!pip install category-encoders


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import pandas as pd
import gc

#pd.set_option('display.max_columns', None)
# Sugerencia
# Extraer fabricante de MAC + Public/private de IP

----------------------------------------------------------------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------------------------------------------------------------
PREPROCESS 1
----------------------------------------------------------------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------------------------------------------------------------

In [15]:
# Lista de CSVs de entrenamiento
df_files = [
    "archive/CIC_IoT_Part_1.csv",
    "archive/CIC_IoT_Part_2.csv",
    "archive/Lab_1.csv",
    "archive/Lab_2.csv",
    "archive/UNSW_IoT_Traces.csv"
]

# Cargar cada dataset por separado en una lista de DataFrames
dfs = []
for f in df_files:
    df_temp = pd.read_csv(f, low_memory=False)
    dfs.append(df_temp)

print(f"Cargados {len(dfs)} datasets:")
for i, df in enumerate(dfs):
    print(f"Dataset {i+1}: {df.shape}")

Cargados 5 datasets:
Dataset 1: (1000000, 89)
Dataset 2: (707918, 89)
Dataset 3: (38125, 89)
Dataset 4: (88692, 89)
Dataset 5: (933833, 89)


In [16]:
# Validación cruzada: Leave-one-dataset-out (4 para train, 1 para test)
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Primero, fit el LabelEncoder en TODOS los datos para manejar clases faltantes
df_all = pd.concat(dfs, ignore_index=True)
le = LabelEncoder()
le.fit(df_all['Type'])

results = []

for test_idx in range(len(dfs)):
    print(f"\n=== Fold {test_idx+1}: Test dataset {test_idx+1} ({df_files[test_idx]}) ===")
    
    # Concatenar los 4 datasets de train
    train_dfs = [dfs[i] for i in range(len(dfs)) if i != test_idx]
    df_train = pd.concat(train_dfs, ignore_index=True)
    df_test = dfs[test_idx]
    
    print(f"Train shape: {df_train.shape}, Test shape: {df_test.shape}")
    
    # Preprocesamiento para train
    cols_to_drop = ['Unnamed: 0', 'FlowID', 'Source', 'SrcIP', 'DstIP', 'Timestamp', 'MAC', 'connection_type', 'DeviceName', 'Type']
    y_train = df_train['Type']
    X_train = df_train.drop(cols_to_drop, axis=1)
    
    # Preprocesamiento para test
    y_test = df_test['Type']
    X_test = df_test.drop(cols_to_drop, axis=1)
    
    # Codificar etiquetas (ya fit en todos los datos)
    y_train_encoded = le.transform(y_train)
    y_test_encoded = le.transform(y_test)
    
    # Escalar
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Entrenar modelo
    model = RandomForestClassifier(n_estimators=50, max_depth=16, random_state=42, n_jobs=2)
    model.fit(X_train_scaled, y_train_encoded)
    
    # Predecir
    y_pred = model.predict(X_test_scaled)
    
    # Evaluar
    acc = accuracy_score(y_test_encoded, y_pred)
    print(f"Accuracy: {acc * 100:.2f}%")
    
    results.append(acc)

# Resultados promedio
print(f"\n=== RESULTADOS PROMEDIO ===")
print(f"Accuracy promedio: {np.mean(results) * 100:.2f}% ± {np.std(results) * 100:.2f}%")
print(f"Resultados por fold: {[f'{r*100:.2f}%' for r in results]}")


=== Fold 1: Test dataset 1 (archive/CIC_IoT_Part_1.csv) ===
Train shape: (1768568, 89), Test shape: (1000000, 89)
Accuracy: 85.97%

=== Fold 2: Test dataset 2 (archive/CIC_IoT_Part_2.csv) ===
Train shape: (2060650, 89), Test shape: (707918, 89)
Accuracy: 62.46%

=== Fold 3: Test dataset 3 (archive/Lab_1.csv) ===
Train shape: (2730443, 89), Test shape: (38125, 89)
Accuracy: 53.15%

=== Fold 4: Test dataset 4 (archive/Lab_2.csv) ===
Train shape: (2679876, 89), Test shape: (88692, 89)
Accuracy: 60.27%

=== Fold 5: Test dataset 5 (archive/UNSW_IoT_Traces.csv) ===
Train shape: (1834735, 89), Test shape: (933833, 89)
Accuracy: 15.29%

=== RESULTADOS PROMEDIO ===
Accuracy promedio: 55.43% ± 22.90%
Resultados por fold: ['85.97%', '62.46%', '53.15%', '60.27%', '15.29%']
